# 03_match_dataset.ipynb
Build `match_features.csv` from team features and historical World Cup match data.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)

In [ ]:
team_df = pd.read_csv("team_features.csv")
matches_df = pd.read_csv("world_cup_matches.csv")

In [ ]:
print("Team Features Shape:", team_df.shape)
print("Matches Shape:", matches_df.shape)

display(team_df.head())
display(matches_df.head())

In [ ]:
team_df["team_name"] = team_df["team_name"].str.strip()
matches_df["home_team"] = matches_df["home_team"].str.strip()
matches_df["away_team"] = matches_df["away_team"].str.strip()

In [ ]:
team_lookup = (
    team_df
    .set_index(["year", "team_name"])
    .to_dict(orient="index")
)

In [ ]:
rows = []

for _, match in matches_df.iterrows():
    year = match["year"]
    home = match["home_team"]
    away = match["away_team"]

    home_key = (year, home)
    away_key = (year, away)

    if home_key not in team_lookup or away_key not in team_lookup:
        continue

    h = team_lookup[home_key]
    a = team_lookup[away_key]

    rows.append({
        "year": year,
        "home_team": home,
        "away_team": away,
        "home_elo": h["elo"],
        "away_elo": a["elo"],
        "home_form_win_rate": h["form_win_rate"],
        "away_form_win_rate": a["form_win_rate"],
        "home_form_gd": h["form_gd_per_game"],
        "away_form_gd": a["form_gd_per_game"],
        "home_form_gf": h["form_gf_per_game"],
        "away_form_gf": a["form_gf_per_game"],
        "home_prev_stage": h["prev_stage_rank"],
        "away_prev_stage": a["prev_stage_rank"],
        "home_wc_apps": h["wc_appearances"],
        "away_wc_apps": a["wc_appearances"],
        "home_age": h["squad_avg_age"],
        "away_age": a["squad_avg_age"],
        "home_exp": h["squad_avg_exp"],
        "away_exp": a["squad_avg_exp"],
        "home_host": h["is_host"],
        "away_host": a["is_host"],
        "home_defending_champ": h["won_last_wc"],
        "away_defending_champ": a["won_last_wc"],
        "home_goals": match["home_goals"],
        "away_goals": match["away_goals"]
    })

match_df = pd.DataFrame(rows)

In [ ]:
def create_result(hg, ag):
    if hg > ag:
        return 0
    elif hg == ag:
        return 1
    return 2

match_df["result"] = match_df.apply(
    lambda x: create_result(x.home_goals, x.away_goals),
    axis=1
)

In [ ]:
match_df["elo_diff"] = match_df["home_elo"] - match_df["away_elo"]
match_df["form_win_diff"] = match_df["home_form_win_rate"] - match_df["away_form_win_rate"]
match_df["form_gd_diff"] = match_df["home_form_gd"] - match_df["away_form_gd"]
match_df["form_gf_diff"] = match_df["home_form_gf"] - match_df["away_form_gf"]
match_df["stage_rank_diff"] = match_df["home_prev_stage"] - match_df["away_prev_stage"]
match_df["age_diff"] = match_df["home_age"] - match_df["away_age"]
match_df["exp_diff"] = match_df["home_exp"] - match_df["away_exp"]
match_df["appearances_diff"] = match_df["home_wc_apps"] - match_df["away_wc_apps"]
match_df["host_diff"] = match_df["home_host"] - match_df["away_host"]
match_df["champion_diff"] = (
    match_df["home_defending_champ"] - match_df["away_defending_champ"]
)

In [ ]:
FEATURES = [
    "elo_diff",
    "form_win_diff",
    "form_gd_diff",
    "form_gf_diff",
    "stage_rank_diff",
    "age_diff",
    "exp_diff",
    "appearances_diff",
    "host_diff",
    "champion_diff"
]

TARGET = "result" 

In [ ]:
print(match_df.shape)
display(match_df[FEATURES + [TARGET]].head())

In [ ]:
missing = match_df.isna().sum().sort_values(ascending=False)
missing[missing > 0]

In [ ]:
match_df = match_df.fillna(0)

Path("data").mkdir(exist_ok=True)

match_df.to_csv("data/match_features.csv", index=False)

print(f"Saved {len(match_df):,} matches")

In [ ]:
match_df["result"].value_counts(normalize=True)